# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---


## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---


In [124]:
# importar librerías
import pandas as pd
import numpy as np

In [125]:
# cargar archivos

orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')

catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')

marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [126]:
# explorar datasets
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [127]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [128]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [129]:
print('Orders:', orders.shape)
print('Catalog:', catalog.shape)
print('Marketing:', marketing.shape)

Orders: (25100, 12)
Catalog: (7, 4)
Marketing: (1620, 5)


In [130]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [131]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [132]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---


**Revisión de valores de fechas**

In [133]:
print("Orders:")
print(orders['fecha_hora_pedido'].head())
print(orders['fecha_hora_pedido'].tail())

print("\nMarketing:")
print(marketing['fecha'].head())
print(marketing['fecha'].tail())

Orders:
0    2025-05-22
1    2025-06-15
2    2025-05-02
3    2025-06-09
4    2025-03-30
Name: fecha_hora_pedido, dtype: object
25095    2025-02-18
25096    2025-04-04
25097    2025-05-13
25098    2025-03-31
25099    2025-06-27
Name: fecha_hora_pedido, dtype: object

Marketing:
0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: fecha, dtype: object
1615    2025-06-29
1616    2025-06-29
1617    2025-06-29
1618    2025-06-29
1619    2025-06-29
Name: fecha, dtype: object


In [134]:
print("Valores únicos en fecha_hora_pedido:", orders['fecha_hora_pedido'].nunique())
print("Valores únicos en fecha:", marketing['fecha'].nunique())

Valores únicos en fecha_hora_pedido: 181
Valores únicos en fecha: 180


In [135]:
print("Rango de fechas de orders:")
print(orders['fecha_hora_pedido'].min(), "a", orders['fecha_hora_pedido'].max())

print("\nRango de fechas de marketing:")
print(marketing['fecha'].min(), "a", marketing['fecha'].max())

Rango de fechas de orders:
2025-01-01 a 2025-06-30

Rango de fechas de marketing:
2025-01-01 a 2025-06-29


In [136]:
orders['fecha_hora_pedido'] = pd.to_datetime(
    orders['fecha_hora_pedido'],
    errors='coerce'
)

marketing['fecha'] = pd.to_datetime(
    marketing['fecha'],
    errors='coerce'
)

**Revisión de valores de fechas**

In [137]:
orders[['cantidad',
        'precio_unitario',
        'monto_descuento',
        'monto_total']].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [138]:
catalog[['costo_unitario']].describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [139]:
marketing[['gasto']].describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [140]:
print("Valores negativos en orders:")
print((orders[['cantidad',
               'precio_unitario',
               'monto_descuento',
               'monto_total']] < 0).sum())

Valores negativos en orders:
cantidad           4
precio_unitario    0
monto_descuento    0
monto_total        4
dtype: int64


In [141]:
print("Valores negativos en catalog:")
print((catalog[['costo_unitario']] < 0).sum())

Valores negativos en catalog:
costo_unitario    0
dtype: int64


In [142]:
print("Valores negativos en marketing:")
print((marketing[['gasto']] < 0).sum())

Valores negativos en marketing:
gasto    0
dtype: int64


---
**📦 Anomalías detectadas** de lo anterior podemos notar que sí hay problemas de calidad en los datos numéricos.

**1. Primer hallazgo**: `cantidad`
Una cantidad negativa no tiene sentido para un pedido normal. Además, que el máximo sea 20,000 cuando el 75% de los pedidos tiene como máximo 2 unidades indica que probablemente existen valores anómalos.

Primero necesitamos saber exactamente cuáles son esos registros.

**2. Segundo hallazgo**: `monto_total` aquí tenemos una señal todavía más fuerte

Tenemos:

mínimo: −493
máximo: 8,840,000
media: 2,070
mediana: 342

**Esta diferencia entre media y mediana es enorme.**


**Para**: `cantidad`

In [143]:
orders[orders['cantidad'] <= 0][[
    'id_pedido',
    'id_usuario',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]]

,id_pedido,id_usuario,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,-1.0,43.50,5.0,-38.50
268,order_268,user_84,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,-1.0,423.53,0.0,-423.53


In [144]:
orders[orders['cantidad'] >= 100][[
    'id_pedido',
    'id_usuario',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]]

,id_pedido,id_usuario,cantidad,precio_unitario,monto_descuento,monto_total
3521,order_3521,user_5812,10000.0,43.14,0.0,431400.0
3522,order_3522,user_3575,10000.0,280.55,0.0,2805500.0
3586,order_3586,user_3380,10000.0,490.35,0.0,4903500.0
3643,order_3643,user_4440,10000.0,238.15,0.0,2381500.0
3656,order_3656,user_884,20000.0,297.66,0.0,5953200.0
3668,order_3668,user_7270,20000.0,348.31,0.0,6966200.0
3689,order_3689,user_6566,10000.0,87.69,0.0,876900.0
3722,order_3722,user_4723,20000.0,442.01,0.0,8840200.0
3726,order_3726,user_2536,20000.0,290.85,0.0,5817000.0
3748,order_3748,user_7096,10000.0,336.93,0.0,3369300.0


**Para**: `monto_total`

In [145]:
orders[orders['monto_total'] < 0][[
    'id_pedido',
    'id_usuario',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]]

,id_pedido,id_usuario,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,-1.0,43.50,5.0,-38.50
268,order_268,user_84,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,-1.0,423.53,0.0,-423.53


**Conclusión**
- Se identificaron cantidades anómalas de 10,000 y 20,000 unidades, incompatibles con el comportamiento esperado del dataset y que generan montos totales desproporcionados. Se consideran registros inválidos y serán excluidos del dataset limpio.
- order_266
    Cantidad: -2
    Precio: 101.31
    Descuento: 10

    El cálculo sería:
    (-2 × 101.31) − 10 = -212.62

    pero monto_total es: -192.62

    Esto indica que en estos registros negativos **la fórmula utilizada para** `monto_total` **podría no seguir la misma lógica**.

    Vamos a comprobar sistemáticamente la consistencia de montos, que justamente es el siguiente punto de nuestra revisión.

In [146]:
# Primero calculamos el total esperado:
orders['monto_total_calculado'] = (
    orders['cantidad'] * orders['precio_unitario']
    - orders['monto_descuento']
)

In [147]:
#Después diferencia entre la columna original vs la calculada
orders['diferencia_monto'] = (
    orders['monto_total'] - orders['monto_total_calculado']
)

In [148]:
orders['diferencia_monto'].describe()

count    25050.000000
mean         0.001563
std          0.154841
min         -0.010000
25%          0.000000
50%          0.000000
75%          0.000000
max         20.000000
Name: diferencia_monto, dtype: float64

In [149]:

# Observar las diferencias significativas:
orders[orders['diferencia_monto'].abs() > 0.01][[
    'id_pedido',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total',
    'monto_total_calculado',
    'diferencia_monto'
]].head(20)


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total,monto_total_calculado,diferencia_monto
2,order_2,2.0,102.99,10.0,195.99,195.98,0.01
24,order_24,2.0,117.99,0.0,235.99,235.98,0.01
35,order_35,2.0,467.34,5.0,929.69,929.68,0.01
41,order_41,2.0,37.77,10.0,65.53,65.54,-0.01
136,order_136,2.0,211.05,15.0,407.09,407.10,-0.01
180,order_180,2.0,451.16,15.0,887.31,887.32,-0.01
261,order_261,2.0,209.33,10.0,408.65,408.66,-0.01
266,order_266,-2.0,101.31,10.0,-192.62,-212.62,20.00
267,order_267,-1.0,43.50,5.0,-38.50,-48.50,10.00
268,order_268,-1.0,497.65,5.0,-492.65,-502.65,10.00


In [150]:
print(
    'Pedidos con diferencia mayor a 1 centavo:',
    (orders['diferencia_monto'].abs() > 0.01).sum()
)

Pedidos con diferencia mayor a 1 centavo: 1149


**Observación**:
- Los registros con cantidades negativas son registros inconsistentes y no deben conservarse como pedidos válidos.
- 1,149 pedidos con diferencia mayor a $0.01... Pero en datos monetarios almacenados como float, también podemos encontrarnos con pequeños errores de representación. Por eso no debemos concluir todavía que existen 1,149 errores de cálculo.

    Hagamos una revisión más precisa:


In [151]:
orders['diferencia_monto'].value_counts().head(20)

 0.000000e+00    18455
-1.000000e-02     2549
 1.000000e-02     2416
 1.000000e-02      171
-1.000000e-02      157
 1.000000e-02      150
 1.000000e-02      148
-1.000000e-02      147
-1.000000e-02      127
 1.000000e-02      126
-1.000000e-02      117
-1.000000e-02       70
 1.000000e-02       68
 7.105427e-15       46
 1.421085e-14       44
-2.842171e-14       35
-7.105427e-15       35
 2.842171e-14       33
-3.552714e-15       30
 3.552714e-15       29
Name: diferencia_monto, dtype: int64

In [152]:
# eliminando las pequeñas imprecisiones de representación de float.
orders['diferencia_monto'].round(2).value_counts().head(20)

 0.00     18791
-0.01      3170
 0.01      3086
 10.00        2
 20.00        1
Name: diferencia_monto, dtype: int64

In [153]:
print(
    'Diferencias mayores a $0.01 después de redondear:',
    (orders['diferencia_monto'].round(2).abs() > 0.01).sum()
)

Diferencias mayores a $0.01 después de redondear: 3


**Resultado clave**:

Después de redondear las diferencias a dos decimales:

Solo **3 registros** presentan una diferencia superior a $0.01.

Y ya sabemos cuáles son:
- order_266 → diferencia \$20
- order_268 → diferencia \$10
- order_267 → diferencia \$10

Los tres corresponden a los pedidos con cantidades negativas que identificamos anteriormente.

- **Hay un cuarto pedido con cantidad negativa**:

    order_269. Pero en ese caso: `-1 × 423.53 − 0 = -423.53` por lo que sí coincide matemáticamente con monto_total.

    Esto es importante: order_269 sigue siendo inválido por su cantidad = -1, pero no tiene además un problema de consistencia de monto.



**En resumen**:
- **4 registros con cantidades negativas**
    order_266, order_267, order_268, order_269
- **10 registros** con cantidades de **10,000** o **20,000**

    Por lo tanto:
    **14 registros serán candidatos a eliminación** durante la etapa de limpieza.

---
**Double check por posible aleración durante exportación de `orders_clean`**:

In [154]:
# Revisar valores actuales de las columnas monetarias y cantidad
orders[['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [155]:
# Revisar algunos registros actuales
orders[['id_pedido', 'cantidad', 'precio_unitario',
        'monto_descuento', 'monto_total']].head(20)

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,2.0,332.69,0.0,665.37
1,order_1,1.0,176.86,5.0,171.86
2,order_2,2.0,102.99,10.0,195.99
3,order_3,1.0,257.87,15.0,242.87
4,order_4,1.0,336.28,0.0,336.28
5,order_5,1.0,179.34,0.0,179.34
6,order_6,2.0,163.59,0.0,327.19
7,order_7,2.0,373.68,0.0,747.35
8,order_8,2.0,477.27,5.0,949.54
9,order_9,1.0,339.30,5.0,334.30


In [156]:
# Revisar específicamente los registros con cantidad = 20
orders[orders['cantidad'] == 20][[
    'id_pedido',
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]].head(20)

,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total


---
**Duplicados**:

**A. Filas completamente duplicadas**:

In [157]:
print("Orders:", orders.duplicated().sum())
print("Catalog:", catalog.duplicated().sum())
print("Marketing:", marketing.duplicated().sum())

Orders: 100
Catalog: 0
Marketing: 0


**B. IDs que deberían ser únicos**:

In [158]:
#Para pedidos:
print(
    "IDs de pedido duplicados:",
    orders['id_pedido'].duplicated().sum()
)

# Para catálogo:
print(
    "Productos duplicados:",
    catalog['nombre_producto'].duplicated().sum()
)

IDs de pedido duplicados: 100
Productos duplicados: 0


**Hallazgo**: los 100 duplicados de orders no son un problema distinto de los IDs duplicados; están relacionados. Así que **vamos a comprobar los duplicados:**

In [159]:
# Comprobar relación de duplicados
orders[orders['id_pedido'].duplicated(keep=False)].sort_values('id_pedido').head(20)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,monto_total_calculado,diferencia_monto
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67,442.68,-0.01
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67,442.68,-0.01
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10,160.10,0.00
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10,160.10,0.00
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08,231.08,0.00
10829,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08,231.08,0.00
10856,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53,72.53,0.00
25044,order_10856,user_1899,2025-03-18,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,1.0,72.53,0.0,72.53,72.53,0.00
11449,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55,55.55,0.00
25083,order_11449,user_1480,2025-03-21,Mexico,desktop,paid_search,Vacuum-Pro-Black,Hogar,1.0,60.55,5.0,55.55,55.55,0.00


In [160]:
# Comprobar cuántos IDs aparecen exactamente dos veces, tres veces, etc.
orders['id_pedido'].value_counts().value_counts().sort_index()

1    24900
2      100
Name: id_pedido, dtype: int64

**📦 Conclusión duplicados**:
- Los 100 duplicados son duplicados reales
    - 24,900 id_pedido aparecen una sola vez.
    - 100 id_pedido aparecen exactamente dos veces.

**✅ Acción**

Podemos eliminar los duplicados conservando una sola copia:
`orders = orders.drop_duplicates()`
Sin embargo, lo haremos más adelante en una sola fase de limpieza, ahora pasemos a las variables categóricas:


---
**Revisión de Variables categóricas**:

In [161]:
# Detectar posibles categorías inconsistentes en orders
for columna in [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto'
]:
    print(f"\n--- {columna} ---")
    print(orders[columna].value_counts(dropna=False))


--- pais ---
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

--- dispositivo ---
desktop    12759
mobile     12321
NaN           20
Name: dispositivo, dtype: int64

--- fuente_referencia ---
social         8428
organic        8329
paid_search    8313
NaN              30
Name: fuente_referencia, dtype: int64

--- nombre_producto ---
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
NaN                       30
Name: nombre_producto, dtype: int64

--- categoria_producto ---
Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: categoria_producto, dtype: int64


In [162]:
# Detectar posibles categorías inconsistentes en marketing
for columna in ['pais', 'canal']:
    print(f"\n--- {columna} ---")
    print(marketing[columna].value_counts(dropna=False))


--- pais ---
Argentina    540
Mexico       540
Colombia     540
Name: pais, dtype: int64

--- canal ---
paid_search    507
social         506
organic        506
NaN            101
Name: canal, dtype: int64


**Hallzgos**: 

1. `pais`en `orders`inconsistencia de mayúsculas/minúsculas

    Colombia     7520
    colombia      823
    
    Mexico       7502
    mexico        865
    
    Argentina    7291
    argentina     799
    
    NaN            300

    - ✅ Solución: Más adelante podemos estandarizar los valores, por ejemplo: `orders['pais'] = orders['pais'].str.capitalize()`


2. `dispositivo`
    - ✅ Categorías consistentes.
    - Tenemos únicamente 20 valores faltantes.


3. `fuente_referencia`
    - ✅ Categorías consistentes.
    - Hay 30 valores faltantes que posteriormente tendremos que tratar.


4. `nombre_producto`
    - 30 valores faltantes. **DEFINICIÓN PENDIENTE**


5. `categoria_producto`
    - Las tres categorías parecen válidas. Pero aquí **tenemos 80 valores faltantes**.
    - Posibible solución: Si tenemos nombre_producto pero falta categoria_producto, podemos recuperar la categoría desde catalog.


6. `pais` en `marketing`
    - ✅ marketing.pais está limpio.
    - Además, esto nos da una referencia para estandarizar orders.pais correctamente.


7. `canal` en `marketing`
    - Tenemos exactamente los tres canales que aparecen en `orders.fuente_referencia`
    - El problema son los **101 valores faltantes**.


**Comprobación antes de limpiar**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

A. Relación producto → categoría
B. ¿Los 30 productos faltantes también tienen categoría?

In [163]:
# Comprobar si el catálogo es consistente
catalog[['nombre_producto', 'categoria_producto']]

,nombre_producto,categoria_producto
0,Laptop-Gaming-16GB,Electrónica
1,Phone-Pro-128GB,Electrónica
2,Tablet-Standard-64GB,Electrónica
3,Blender-XL-Red,Hogar
4,Vacuum-Pro-Black,Hogar
5,Sneakers-Urban-42,Moda
6,Jacket-Winter-M,Moda


In [164]:
# Verificar si los 80 pedidos con categoría faltante sí tienen producto registrado.
orders[orders['categoria_producto'].isna()][[
    'id_pedido',
    'nombre_producto',
    'categoria_producto'
]].head(20)

,id_pedido,nombre_producto,categoria_producto
44,order_44,NaN,NaN
45,order_45,NaN,NaN
46,order_46,NaN,NaN
47,order_47,NaN,NaN
48,order_48,NaN,NaN
49,order_49,NaN,NaN
50,order_50,NaN,NaN
51,order_51,NaN,NaN
52,order_52,NaN,NaN
53,order_53,NaN,NaN


---
**Hallazgo**:

Tenemos:

- 30 `nombre_producto` faltantes
- 80 `categoria_producto` faltantes

Por lo tanto, necesariamente hay:

- 80 − 30 = 50 registros que sí tienen: `producto conocido` + `categoría faltante`

- Esos **50 registros** sí podrían **recuperarse** desde catalog.

In [165]:
# Listar pedidos con Producto conocido + categoría faltante
print("Producto conocido + categoría faltante:")

orders[
    orders['nombre_producto'].notna() &
    orders['categoria_producto'].isna()
][[
    'id_pedido',
    'nombre_producto',
    'categoria_producto'
]]

Producto conocido + categoría faltante:


,id_pedido,nombre_producto,categoria_producto
74,order_74,Sneakers-Urban-42,NaN
75,order_75,Sneakers-Urban-42,NaN
76,order_76,Laptop-Gaming-16GB,NaN
77,order_77,Vacuum-Pro-Black,NaN
78,order_78,Phone-Pro-128GB,NaN
79,order_79,Sneakers-Urban-42,NaN
80,order_80,Vacuum-Pro-Black,NaN
81,order_81,Tablet-Standard-64GB,NaN
82,order_82,Sneakers-Urban-42,NaN
83,order_83,Sneakers-Urban-42,NaN


In [166]:
# Comprobar que haya cero pedidos con Producto faltante + categoría conocida 
print("Producto faltante + categoría conocida:")

orders[
    orders['nombre_producto'].isna() &
    orders['categoria_producto'].notna()
][[
    'id_pedido',
    'nombre_producto',
    'categoria_producto'
]]

Producto faltante + categoría conocida:


,id_pedido,nombre_producto,categoria_producto


**Hallazgos**:
1. Tenemos **50 categorías recuperables**
    - No necesitan eliminarse.
    - Podemos completar `categoria_producto` utilizando `catalog`.
2. Los otros **30** son casos realmente **incompletos**
    - Los dejaremos como **faltantes por ahora** y posteriormente decidiremos si:
        - **se conservan** para determinados análisis;
        - o **se excluyen** de los análisis que necesiten información del producto.

**Revisión de los otros valores faltantes**: 
Ahora tenemos que investigar los faltantes de:
- `orders`
    - pais → 300
    - dispositivo → 20
    - fuente_referencia → 30
- `marketing`
    - canal → 101

In [167]:
# Comprobar si existe alguna relación que permita recuperarlos
orders[orders['pais'].isna()][[
    'id_pedido',
    'id_usuario',
    'fecha_hora_pedido',
    'dispositivo',
    'fuente_referencia',
    'nombre_producto',
    'categoria_producto',
    'monto_total'
]].head(20)

,id_pedido,id_usuario,fecha_hora_pedido,dispositivo,fuente_referencia,nombre_producto,categoria_producto,monto_total
124,order_124,user_6671,2025-04-12,mobile,organic,Blender-XL-Red,Hogar,293.44
125,order_125,user_5263,2025-05-30,mobile,organic,Tablet-Standard-64GB,Electronica,133.13
126,order_126,user_4952,2025-04-19,desktop,paid_search,Phone-Pro-128GB,Electronica,354.45
127,order_127,user_2619,2025-06-26,desktop,paid_search,Tablet-Standard-64GB,Electronica,319.53
128,order_128,user_2444,2025-05-06,desktop,social,Vacuum-Pro-Black,Hogar,436.86
129,order_129,user_7157,2025-04-17,desktop,paid_search,Sneakers-Urban-42,Moda,357.88
130,order_130,user_3279,2025-02-06,desktop,paid_search,Vacuum-Pro-Black,Hogar,114.83
131,order_131,user_2914,2025-05-05,mobile,organic,Blender-XL-Red,Hogar,107.84
132,order_132,user_2173,2025-02-03,desktop,organic,Tablet-Standard-64GB,Electronica,383.84
133,order_133,user_4038,2025-05-12,mobile,social,Vacuum-Pro-Black,Hogar,691.69


In [168]:
orders[orders['dispositivo'].isna()][[
    'id_pedido',
    'id_usuario',
    'pais',
    'fuente_referencia',
    'nombre_producto',
    'monto_total'
]]

,id_pedido,id_usuario,pais,fuente_referencia,nombre_producto,monto_total
24,order_24,user_458,Colombia,organic,Sneakers-Urban-42,235.99
25,order_25,user_540,Argentina,social,Vacuum-Pro-Black,178.50
26,order_26,user_1951,Colombia,organic,Blender-XL-Red,316.52
27,order_27,user_5410,colombia,paid_search,Phone-Pro-128GB,893.96
28,order_28,user_1489,Mexico,social,Phone-Pro-128GB,50.70
29,order_29,user_5197,Colombia,social,Phone-Pro-128GB,381.57
30,order_30,user_5386,Mexico,social,Vacuum-Pro-Black,173.74
31,order_31,user_5766,Colombia,paid_search,Vacuum-Pro-Black,480.74
32,order_32,user_4400,Mexico,social,Tablet-Standard-64GB,117.69
33,order_33,user_5070,Argentina,paid_search,Tablet-Standard-64GB,86.82


In [169]:
orders[orders['fuente_referencia'].isna()][[
    'id_pedido',
    'id_usuario',
    'pais',
    'dispositivo',
    'nombre_producto',
    'monto_total'
]]

,id_pedido,id_usuario,pais,dispositivo,nombre_producto,monto_total
44,order_44,user_2899,Colombia,desktop,NaN,313.38
45,order_45,user_6196,Mexico,mobile,NaN,698.12
46,order_46,user_5815,Mexico,mobile,NaN,349.31
47,order_47,user_406,Colombia,mobile,NaN,953.56
48,order_48,user_6187,Colombia,desktop,NaN,275.93
49,order_49,user_7225,Argentina,desktop,NaN,867.53
50,order_50,user_1840,Mexico,desktop,NaN,490.38
51,order_51,user_5998,Mexico,desktop,NaN,765.89
52,order_52,user_5679,Argentina,mobile,NaN,830.01
53,order_53,user_5936,Argentina,desktop,NaN,472.98


**Hallazgos**: 
1. `pais`: **300 valores faltantes**
    Los registros que muestras tienen información suficiente de: usuario, dispositivo, fuente, producto, categoría y monto.
   
    Pero **no existe otro campo** dentro de ese pedido **que permita determinar de forma confiable el país**.

   **Conclusión:**

    Los 300 pais = NaN no son recuperables directamente a partir de la información del pedido.

    No vamos a inventar el país ni a asignarlo basándonos en patrones como dispositivo, producto o fuente.

2. `dispositivo`: **20 valores faltantes**

    Los 20 pedidos tienen país, fuente y producto, pero dispositivo está vacío.

    **Conclusión**

    Los 20 valores faltantes tampoco son recuperables de manera confiable.


3. `fuente_referencia`: **30 valores faltantes**

    Los 30 pedidos tienen: país, dispositivo y producto... pero **no tienen fuente de adquisición**.

    **Conclusión**

    Los 30 valores faltantes deben tratarse como desconocidos.

**Revisar marketing.canal**:


In [170]:
# Revisión de canales faltantes
marketing[marketing['canal'].isna()][[
    'fecha',
    'pais',
    'id_campaña',
    'canal',
    'gasto'
]].head(30)

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
103,2025-01-12,Colombia,paid_search_Colombia,NaN,1583.33
104,2025-01-12,Colombia,social_Colombia,NaN,712.64
105,2025-01-12,Argentina,organic_Argentina,NaN,1860.40
106,2025-01-12,Argentina,paid_search_Argentina,NaN,885.06
107,2025-01-12,Argentina,social_Argentina,NaN,2253.77


In [171]:
# Listar campañas únicas
marketing[marketing['canal'].isna()]['id_campaña'].unique()

array(['social_Argentina', 'organic_Mexico', 'paid_search_Mexico',
       'social_Mexico', 'organic_Colombia', 'paid_search_Colombia',
       'social_Colombia', 'organic_Argentina', 'paid_search_Argentina'],
      dtype=object)

**Hallazgos**: 
1. `marketing.canal`: problema **completamente recuperable**
    - El propio id_campaña contiene el canal.
    - Esto nos permite **recuperar los 101 valores faltantes sin hacer una imputación arbitraria**.

**Continuar revisión**: 

- Primero: productos de orders que no existen en catalog
- Después: categorías de orders que no existen en catalog
- Y finalmente, canales de orders que no existen en marketing

In [172]:
# Verificar productos de orders que no existen en catalog
productos_orders = orders['nombre_producto'].dropna().unique()
productos_catalogo = catalog['nombre_producto'].unique()

productos_no_catalogo = set(productos_orders) - set(productos_catalogo)

print("Productos de orders no encontrados en catalog:")
print(productos_no_catalogo)

Productos de orders no encontrados en catalog:
set()


In [173]:
# Verificar categorías de orders que no existen en catalog
categorias_orders = orders['categoria_producto'].dropna().unique()
categorias_catalogo = catalog['categoria_producto'].unique()

categorias_no_catalogo = set(categorias_orders) - set(categorias_catalogo)

print("Categorías de orders no encontradas en catalog:")
print(categorias_no_catalogo)

Categorías de orders no encontradas en catalog:
{'Electronica'}


In [174]:
# Verificar fuentes de referencia de orders contra canales de marketing
fuentes_orders = orders['fuente_referencia'].dropna().unique()
canales_marketing = marketing['canal'].dropna().unique()

fuentes_sin_correspondencia = set(fuentes_orders) - set(canales_marketing)

print("Fuentes de orders sin correspondencia en marketing:")
print(fuentes_sin_correspondencia)

Fuentes de orders sin correspondencia en marketing:
set()


**Hallazgos**: 
1. Productos: todo correcto ✅
2. Categorías: encontramos una inconsistencia ⚠️
   - Obtuvimos {'**Electronica**'}
   - Mientras que en `catalog` la categoría aparece como: **Electrónica**
   - ✅ Solución: **Estandarizaremos** Electronica → Electrónica.
3. Fuentes de referencia: todo correcto ✅

---
**🧹✨ Plan de limpieza**:
- `orders`
    | Problema                                            |               Registros | Decisión                      |
| --------------------------------------------------- | ----------------------: | ----------------------------- |
| Duplicados completos                                |                     100 | **Eliminar**                  |
| `cantidad < 0`                                      |                       4 | **Eliminar**                  |
| `cantidad` = 10,000/20,000                          |                      10 | **Eliminar**                  |
| `pais` en minúsculas                                |                   2,487 | **Estandarizar**              |
| `pais` faltante                                     |                     300 | **Conservar como NaN**        |
| `dispositivo` faltante                              |                      20 | **Conservar como NaN**        |
| `fuente_referencia` faltante                        |                      30 | **Conservar como NaN**        |
| `nombre_producto` faltante                          |                      30 | **Conservar como NaN**        |
| `categoria_producto` faltante con producto conocido |                      50 | **Completar desde `catalog`** |
| `categoria_producto` faltante sin producto          |                      30 | **Conservar como NaN**        |
| `Electronica`                                       | Categoría inconsistente | **Cambiar a `Electrónica`**   |


- `catalog`
    | Problema          |    Registros | Decisión     |
| ----------------- | -----------: | ------------ |
| Duplicados        |            0 | No modificar |
| Nulos             |            0 | No modificar |
| Valores negativos |            0 | No modificar |
| Categorías        | Consistentes | No modificar |


- `marketing`
    | Problema          |    Registros | Decisión                         |
| ----------------- | -----------: | -------------------------------- |
| Duplicados        |            0 | No modificar                     |
| Nulos en `canal`  |          101 | **Recuperar desde `id_campaña`** |
| Valores negativos |            0 | No modificar                     |
| Países            | Consistentes | No modificar                     |




**Pero hay una cuestión que quiero resolver antes de limpiar**

- Tenemos **100 duplicados** y **14 registros** numéricamente **anómalos**.

- Necesitamos saber si alguno de esos **14 registros está dentro de los 100 duplicados**.

    Porque si hubiera, por ejemplo:

    - 2 registros negativos que además están duplicados **no queremos contabilizar esos problemas** como: `100 + 14 = 114 filas` sin comprobarlo.

- Y también necesitamos **saber si los duplicados están involucrados en los demás problemas**.

Por eso hagamos una última comprobación antes de ejecutar la limpieza:

In [175]:
# Revisar si los registros con cantidades inválidas están duplicados
pedidos_invalidos = orders[
    (orders['cantidad'] < 0) |
    (orders['cantidad'] >= 10000)
]

print("Registros numéricamente inválidos:", len(pedidos_invalidos))
print("IDs inválidos duplicados:", pedidos_invalidos['id_pedido'].duplicated().sum())

print("\nIDs de registros inválidos:")
print(pedidos_invalidos['id_pedido'].tolist())

Registros numéricamente inválidos: 14
IDs inválidos duplicados: 0

IDs de registros inválidos:
['order_266', 'order_267', 'order_268', 'order_269', 'order_3521', 'order_3522', 'order_3586', 'order_3643', 'order_3656', 'order_3668', 'order_3689', 'order_3722', 'order_3726', 'order_3748']


In [176]:
# Comprobar si los registros duplicados contienen valores numéricos inválidos
duplicados = orders[
    orders['id_pedido'].duplicated(keep=False)
]

duplicados_invalidos = duplicados[
    (duplicados['cantidad'] < 0) |
    (duplicados['cantidad'] >= 10000)
]

print("Registros duplicados con cantidades inválidas:", len(duplicados_invalidos))
print("IDs:", duplicados_invalidos['id_pedido'].unique())

Registros duplicados con cantidades inválidas: 0
IDs: []


**Hallazgos**:
1. Los **14 registros numéricamente inválidos no están entre los duplicados**, así que las dos correcciones son independientes.

**Resumen final del diagnóstico**
- `orders` — 25,100 registros iniciales
    | Problema                                          | Cantidad | Acción                       |
| ------------------------------------------------- | -------: | ---------------------------- |
| Duplicados por `id_pedido`                        |      100 | Eliminar                     |
| `cantidad` negativa                               |        4 | Eliminar                     |
| `cantidad` extrema (10,000–20,000)                |       10 | Eliminar                     |
| `pais` en minúsculas                              |    2,487 | Estandarizar                 |
| `pais` faltante                                   |      300 | Conservar                    |
| `dispositivo` faltante                            |       20 | Conservar                    |
| `fuente_referencia` faltante                      |       30 | Conservar                    |
| Producto faltante                                 |       30 | Conservar                    |
| Categoría faltante con producto conocido          |       50 | Completar desde catálogo     |
| Categoría faltante sin producto                   |       30 | Conservar                    |
| `Electronica`                                     |        — | Estandarizar a `Electrónica` |
| Diferencias de monto > $0.01 después de redondear |        3 | Revisar                      |

- `catalog`
    Está limpio: 7 × 4, sin duplicados, nulos ni valores numéricos inválidos.

- `marketing`
    Está estructuralmente limpio salvo los 101 valores faltantes de canal, que podemos recuperar mediante id_campaña.

---
**🧹✨ Empezamos la limpieza**: 

1. Crear copias de respaldo.
2. Eliminar duplicados de orders.
3. Eliminar los 14 registros numéricamente inválidos.
4. Estandarizar países.
5. Completar categorías utilizando catalog.
6. Estandarizar categorías.
7. Recuperar marketing.canal.
8. Validar nuevamente los tres datasets.

In [177]:
# Crear copias de los datasets originales
orders_clean = orders.copy()
catalog_clean = catalog.copy()
marketing_clean = marketing.copy()

In [178]:
# Eliminar pedidos duplicados conservando la primera aparición
filas_antes = len(orders_clean)

orders_clean = orders_clean.drop_duplicates(subset='id_pedido', keep='first')

filas_despues = len(orders_clean)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

Filas antes: 25100
Filas después: 25000
Filas eliminadas: 100


In [179]:
# Eliminar pedidos con cantidades numéricamente inválidas
filas_antes = len(orders_clean)

orders_clean = orders_clean[
    (orders_clean['cantidad'] > 0) &
    (orders_clean['cantidad'] < 10000)
]

filas_despues = len(orders_clean)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

Filas antes: 25000
Filas después: 24936
Filas eliminadas: 64


In [180]:
# Reconstruir orders_clean y eliminar únicamente duplicados y cantidades inválidas conocidas
orders_clean = orders.copy()

orders_clean = orders_clean.drop_duplicates(subset='id_pedido', keep='first')

condicion_invalida = (
    orders_clean['cantidad'].notna() &
    (
        (orders_clean['cantidad'] <= 0) |
        (orders_clean['cantidad'] >= 10000)
    )
)

print("Registros con cantidad inválida:", condicion_invalida.sum())

orders_clean = orders_clean[~condicion_invalida]

print("Filas finales:", len(orders_clean))
print("Valores faltantes en cantidad:", orders_clean['cantidad'].isna().sum())

Registros con cantidad inválida: 14
Filas finales: 24986
Valores faltantes en cantidad: 50


In [181]:
# Revisar si podemos recuperar cantidad a partir de los montos
cantidad_faltante = orders_clean[orders_clean['cantidad'].isna()].copy()

cantidad_faltante['cantidad_calculada'] = (
    cantidad_faltante['monto_total'] +
    cantidad_faltante['monto_descuento']
) / cantidad_faltante['precio_unitario']

print("Registros con cantidad faltante:", len(cantidad_faltante))

print("\nCantidad calculada:")
print(cantidad_faltante['cantidad_calculada'].describe())

print("\nPrimeros registros:")
print(
    cantidad_faltante[
        ['id_pedido', 'precio_unitario', 'monto_descuento',
         'monto_total', 'cantidad_calculada']
    ].head(20)
)

Registros con cantidad faltante: 50

Cantidad calculada:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: cantidad_calculada, dtype: float64

Primeros registros:
   id_pedido  precio_unitario  monto_descuento  monto_total  \
74  order_74              NaN              NaN       595.85   
75  order_75              NaN              NaN       458.15   
76  order_76              NaN              NaN       319.75   
77  order_77              NaN              NaN       227.55   
78  order_78              NaN              NaN       432.39   
79  order_79              NaN              NaN       527.15   
80  order_80              NaN              NaN       711.18   
81  order_81              NaN              NaN        41.08   
82  order_82              NaN              NaN       431.55   
83  order_83              NaN              NaN       279.35   
84  order_84              NaN              NaN       785.89   
85  order_85         

In [182]:
# Estandarizar nombres de países en orders
orders_clean['pais'] = orders_clean['pais'].replace({
    'mexico': 'Mexico',
    'colombia': 'Colombia',
    'argentina': 'Argentina'
})

print(orders_clean['pais'].value_counts(dropna=False))

Mexico       8336
Colombia     8302
Argentina    8052
NaN           296
Name: pais, dtype: int64


In [183]:
# Completar categorías faltantes utilizando el catálogo de productos
orders_clean = orders_clean.merge(
    catalog_clean[['nombre_producto', 'categoria_producto']],
    on='nombre_producto',
    how='left',
    suffixes=('', '_catalogo')
)

print("Categorías faltantes antes:", orders_clean['categoria_producto'].isna().sum())
print("Categorías disponibles en catálogo:", orders_clean['categoria_producto_catalogo'].notna().sum())

Categorías faltantes antes: 80
Categorías disponibles en catálogo: 24956


In [184]:
# Completar categorías faltantes con la información del catálogo
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].fillna(
    orders_clean['categoria_producto_catalogo']
)

# Eliminar la columna auxiliar del catálogo
orders_clean = orders_clean.drop(columns='categoria_producto_catalogo')

print("Categorías faltantes después:", orders_clean['categoria_producto'].isna().sum())

Categorías faltantes después: 30


In [185]:
# Estandarizar nombres de categorías en orders
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].replace({
    'Electronica': 'Electrónica'
})

print(orders_clean['categoria_producto'].value_counts(dropna=False))

Hogar          8355
Moda           8324
Electrónica    8277
NaN              30
Name: categoria_producto, dtype: int64


In [186]:
# Revisar valores únicos de dispositivo y fuente de referencia
print('--- dispositivo ---')
print(orders_clean['dispositivo'].value_counts(dropna=False))

print('\n--- fuente_referencia ---')
print(orders_clean['fuente_referencia'].value_counts(dropna=False))

--- dispositivo ---
desktop    12702
mobile     12264
NaN           20
Name: dispositivo, dtype: int64

--- fuente_referencia ---
social         8398
organic        8285
paid_search    8273
NaN              30
Name: fuente_referencia, dtype: int64


In [187]:
# Revisar valores faltantes después de la limpieza de orders
faltantes_orders = orders_clean.isna().sum()

print(faltantes_orders[faltantes_orders > 0])

pais                     296
dispositivo               20
fuente_referencia         30
nombre_producto           30
categoria_producto        30
cantidad                  50
precio_unitario           50
monto_descuento           50
monto_total_calculado     50
diferencia_monto          50
dtype: int64


In [188]:
# Contar registros que contienen al menos un valor faltante
print(
    'Registros con al menos un valor faltante:',
    orders_clean.isna().any(axis=1).sum()
)

Registros con al menos un valor faltante: 396


---
**Ahora pasemos a `catalog`**

`catalog` es mucho más pequeño y, por lo que vimos inicialmente, no tiene valores faltantes.

Primero hagamos una revisión general antes de modificarlo.

In [189]:
# Revisar calidad de datos de catalog
print(catalog.info())

print('\nValores faltantes:')
print(catalog.isna().sum())

print('\nDuplicados:')
print(catalog.duplicated().sum())

print('\nValores únicos por columna:')
for columna in catalog.columns:
    print(f'\n--- {columna} ---')
    print(catalog[columna].value_counts(dropna=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
None

Valores faltantes:
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

Duplicados:
0

Valores únicos por columna:

--- nombre_producto ---
Phone-Pro-128GB         1
Vacuum-Pro-Black        1
Laptop-Gaming-16GB      1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Jacket-Winter-M         1
Sneakers-Urban-42       1
Name: nombre_producto, dtype: int64

--- categoria_producto ---
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

--- costo

In [190]:
# Crear copia limpia del catálogo
catalog_clean = catalog.copy()

print('Filas:', len(catalog_clean))
print('Columnas:', catalog_clean.shape[1])
print('Valores faltantes:', catalog_clean.isna().sum().sum())
print('Duplicados:', catalog_clean.duplicated().sum())

Filas: 7
Columnas: 4
Valores faltantes: 0
Duplicados: 0


---
**Ahora pasemos a `marketing`**

Aquí ya conocemos un problema importante:

- 1,620 registros.
- 101 valores faltantes en canal.
- fecha inicialmente era object, pero ya la convertimos a fecha.
- No hay valores faltantes en pais, id_campaña ni gasto.
- No hay gastos negativos.
- Las combinaciones de campaña siguen un patrón consistente:
- organic_Pais
- paid_search_Pais
- social_Pais

Así que vamos a hacer primero la misma revisión estructural que hicimos con `catalog`.

In [191]:
# Revisar calidad de datos de marketing
print(marketing.info())

print('\nValores faltantes:')
print(marketing.isna().sum())

print('\nDuplicados:')
print(marketing.duplicated().sum())

print('\nValores únicos por columna:')
for columna in marketing.columns:
    print(f'\n--- {columna} ---')
    print(marketing[columna].value_counts(dropna=False).head(20))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB
None

Valores faltantes:
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

Duplicados:
0

Valores únicos por columna:

--- fecha ---
2025-06-25    9
2025-03-31    9
2025-01-11    9
2025-06-29    9
2025-02-25    9
2025-04-11    9
2025-05-26    9
2025-01-22    9
2025-03-08    9
2025-04-22    9
2025-06-06    9
2025-02-02    9
2025-03-19    9
2025-05-03    9
2025-06-17    9
2025-02-13    9
2025-03-30    9
2025-05-14    9
2025-01-10 

In [192]:
# Recuperar canal a partir del identificador de campaña
mapa_canal = {
    'organic_Mexico': 'organic',
    'paid_search_Mexico': 'paid_search',
    'social_Mexico': 'social',
    'organic_Colombia': 'organic',
    'paid_search_Colombia': 'paid_search',
    'social_Colombia': 'social',
    'organic_Argentina': 'organic',
    'paid_search_Argentina': 'paid_search',
    'social_Argentina': 'social'
}

marketing_clean = marketing.copy()

marketing_clean['canal'] = marketing_clean['canal'].fillna(
    marketing_clean['id_campaña'].map(mapa_canal)
)

print('Canales faltantes después:', marketing_clean['canal'].isna().sum())
print('\nDistribución final:')
print(marketing_clean['canal'].value_counts(dropna=False))

Canales faltantes después: 0

Distribución final:
paid_search    540
social         540
organic        540
Name: canal, dtype: int64


---
**📦 Validación final de los 3 datasets**:

In [193]:
# Validación final de calidad de los tres datasets

print('===== ORDERS =====')
print('Filas:', orders_clean.shape[0])
print('Columnas:', orders_clean.shape[1])
print('Duplicados:', orders_clean.duplicated().sum())
print('Faltantes:', orders_clean.isna().sum().sum())
print('Tipos de datos:')
print(orders_clean.dtypes)

print('\n===== CATALOG =====')
print('Filas:', catalog_clean.shape[0])
print('Columnas:', catalog_clean.shape[1])
print('Duplicados:', catalog_clean.duplicated().sum())
print('Faltantes:', catalog_clean.isna().sum().sum())
print('Tipos de datos:')
print(catalog_clean.dtypes)

print('\n===== MARKETING =====')
print('Filas:', marketing_clean.shape[0])
print('Columnas:', marketing_clean.shape[1])
print('Duplicados:', marketing_clean.duplicated().sum())
print('Faltantes:', marketing_clean.isna().sum().sum())
print('Tipos de datos:')
print(marketing_clean.dtypes)

===== ORDERS =====
Filas: 24986
Columnas: 14
Duplicados: 0
Faltantes: 656
Tipos de datos:
id_pedido                        object
id_usuario                       object
fecha_hora_pedido        datetime64[ns]
pais                             object
dispositivo                      object
fuente_referencia                object
nombre_producto                  object
categoria_producto               object
cantidad                        float64
precio_unitario                 float64
monto_descuento                 float64
monto_total                     float64
monto_total_calculado           float64
diferencia_monto                float64
dtype: object

===== CATALOG =====
Filas: 7
Columnas: 4
Duplicados: 0
Faltantes: 0
Tipos de datos:
nombre_producto        object
categoria_producto     object
costo_unitario        float64
proveedor              object
dtype: object

===== MARKETING =====
Filas: 1620
Columnas: 5
Duplicados: 0
Faltantes: 0
Tipos de datos:
fecha         datetime64[ns

In [194]:
# Validación final de variables numéricas en orders

print('Cantidades negativas:', (orders_clean['cantidad'] < 0).sum())
print('Precios negativos:', (orders_clean['precio_unitario'] < 0).sum())
print('Descuentos negativos:', (orders_clean['monto_descuento'] < 0).sum())
print('Totales negativos:', (orders_clean['monto_total'] < 0).sum())

print('\nCantidades iguales a cero:', (orders_clean['cantidad'] == 0).sum())
print('Precios iguales a cero:', (orders_clean['precio_unitario'] == 0).sum())

Cantidades negativas: 0
Precios negativos: 0
Descuentos negativos: 0
Totales negativos: 0

Cantidades iguales a cero: 0
Precios iguales a cero: 0


---
**📦 Resultado final**:
| Dataset           |  Filas | Columnas | Duplicados | Faltantes |
| ----------------- | -----: | -------: | ---------: | --------: |
| `orders_clean`    | 24,986 |       14 |          0 |       656 |
| `catalog_clean`   |      7 |        4 |          0 |         0 |
| `marketing_clean` |  1,620 |        5 |          0 |         0 |


**Una observación antes de seguir**

`orders_clean` tiene ahora 14 columnas, porque todavía conservamos: `monto_total_calculado` y `diferencia_monto`.

Procedo a eliminarlas ahora:


In [195]:
# Eliminar columnas auxiliares utilizadas para validar los montos
orders_clean = orders_clean.drop(
    columns=['monto_total_calculado', 'diferencia_monto']
)

print('Columnas finales:', orders_clean.shape[1])
print(orders_clean.columns.tolist())

Columnas finales: 12
['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [196]:
# exportar datasets
# orders.to_csv('orders_clean.csv', index=False)
# catalog.to_csv('catalog_clean.csv', index=False)
# marketing.to_csv('marketing_clean.csv', index=False)
#Como utilicé _clean para cada dataset limpio modifico esta celda como se muestra a continuación:

# Exportar datasets limpios para las siguientes etapas del proyecto
orders_clean.to_csv('orders_clean.csv', index=False)
catalog_clean.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

print('Datasets limpios exportados correctamente.')

Datasets limpios exportados correctamente.


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?

    Será la suma de `monto_total` de `orders_clean`.

    **Revenue total = $9,643,909.56**

- ¿Cuál es el costo total?
    Necesitamos relacionar `orders_clean` con `catalog_clean` mediante `nombre_producto` para obtener `costo_unitario`.

    Entonces `costo_producto = cantidad × costo_unitario`

- ¿Cuánto se ha invertido en marketing?
    Será la suma de gasto de marketing_clean.

    
- ¿El negocio es rentable? (calcular profit)  
    `Profit = Revenue − Costo de productos − Gasto de marketing`
---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?

    Ticket promedio por orden: **$385.97**
- ¿Cuál es la cantidad promedio de productos por orden?

    Cantidad promedio de productos por orden: **1.50**
- ¿Cuál es el producto más vendido?

    **Vacuum-Pro-Black: 6284 unidades**
- ¿Cuánto se ha gastado en marketing por canal?
    | Canal       |             Gasto |
| ----------- | ----------------: |
| Social      |   **＄976,818.37** |
| Organic     |   **＄972,650.96** |
| Paid Search |   **＄922,374.20** |
| **Total**   | **＄2,871,843.53** |



In [197]:
# Calcular revenue total del negocio
revenue_total = orders_clean['monto_total'].sum()

print(f'Revenue total: ${revenue_total:,.2f}')

Revenue total: $9,643,909.56


In [198]:
# Incorporar el costo unitario del catálogo a los pedidos
orders_profit = orders_clean.merge(
    catalog_clean[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

print('Filas antes del merge:', len(orders_clean))
print('Filas después del merge:', len(orders_profit))
print('Costos unitarios faltantes:', orders_profit['costo_unitario'].isna().sum())

Filas antes del merge: 24986
Filas después del merge: 24986
Costos unitarios faltantes: 30


In [199]:
# Calcular el costo de productos por pedido
orders_profit['costo_producto'] = (
    orders_profit['cantidad'] *
    orders_profit['costo_unitario']
)

print('Costos de producto calculados:', orders_profit['costo_producto'].notna().sum())
print('Costos de producto faltantes:', orders_profit['costo_producto'].isna().sum())

print('\nCosto total calculable:')
print(f"${orders_profit['costo_producto'].sum():,.2f}")

Costos de producto calculados: 24906
Costos de producto faltantes: 80

Costo total calculable:
$3,828,869.01


**Nota:**

Tenemos:
- Revenue: ＄9,643,909.56
- Costo de producto calculable: ＄3,828,869.01
- 80 pedidos sin costo calculable
    - 50 por cantidad faltante.
    - 30 por nombre_producto/costo_unitario faltante.

- No debemos tratar esos 80 costos como ＄0, porque eso inflaría artificialmente el profit.

In [200]:
# Calcular inversión total en marketing
marketing_total = marketing_clean['gasto'].sum()

print(f'Gasto total en marketing: ${marketing_total:,.2f}')

Gasto total en marketing: $2,871,843.53


**Nota:**

Ahora podemos calcular el **profit calculable**.

Pero antes, una precisión: como existen 80 pedidos cuyo costo no podemos determinar, este profit debe presentarse como **profit calculable**, no como **profit definitivo**. Así evitamos afirmar que conocemos una rentabilidad que los datos no permiten calcular completamente.

In [201]:
# Calcular profit utilizando los costos de producto disponibles
profit_calculable = (
    revenue_total -
    orders_profit['costo_producto'].sum() -
    marketing_total
)

print(f'Profit calculable: ${profit_calculable:,.2f}')

Profit calculable: $2,943,197.02


In [202]:
# Calcular margen de profit calculable
margen_profit = profit_calculable / revenue_total * 100

print(f'Margen de profit calculable: {margen_profit:.2f}%')

Margen de profit calculable: 30.52%


**📈 Parte 2: Comportamiento de ventas**

In [203]:
# Calcular ticket promedio por orden
ticket_promedio = orders_clean['monto_total'].mean()

print(f'Ticket promedio por orden: ${ticket_promedio:,.2f}')

Ticket promedio por orden: $385.97


In [204]:
# Calcular cantidad promedio de productos por orden
cantidad_promedio = orders_clean['cantidad'].mean()

print(f'Cantidad promedio de productos por orden: {cantidad_promedio:.2f}')

Cantidad promedio de productos por orden: 1.50


In [205]:
# Calcular unidades vendidas por producto
ventas_producto = (
    orders_clean
    .groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

print('Unidades vendidas por producto:')
print(ventas_producto)

Unidades vendidas por producto:
nombre_producto
Vacuum-Pro-Black        6284.0
Blender-XL-Red          6279.0
Jacket-Winter-M         6256.0
Sneakers-Urban-42       6172.0
Laptop-Gaming-16GB      4198.0
Tablet-Standard-64GB    4153.0
Phone-Pro-128GB         4140.0
Name: cantidad, dtype: float64


In [206]:
# Calcular gasto total de marketing por canal
marketing_por_canal = (
    marketing_clean
    .groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

print('Gasto de marketing por canal:')
print(marketing_por_canal)

Gasto de marketing por canal:
canal
social         976818.37
organic        972650.96
paid_search    922374.20
Name: gasto, dtype: float64


**Notas**:
- Importante: todavía no concluiría que `social` es el **mejor canal**. Hasta ahora sabemos cuánto se gastó en cada canal, pero no cuánto revenue generó cada uno.

- El costo total de $3.83 M es **calculable, no necesariamente el costo total real** del negocio, porque **tenemos 80 registros sin costo de producto**. Conviene conservar esa distinción en el análisis final.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [207]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [208]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [209]:
# Revisar eventos disponibles y su frecuencia
print(events['nombre_evento'].value_counts())

first_visit         29957
add_to_cart         24157
select_item         23887
begin_checkout      17971
add_payment_info    12018
purchase            12010
Name: nombre_evento, dtype: int64


In [210]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
WHERE nombre_evento IN (
    'first_visit',
    'add_to_cart',
    'select_item',
    'begin_checkout',
    'add_payment_info',
    'purchase'
)
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'add_to_cart' THEN 2
        WHEN 'select_item' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [211]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    WHERE nombre_evento IN (
        'first_visit',
        'add_to_cart',
        'select_item',
        'begin_checkout',
        'add_payment_info',
        'purchase'
    )
    GROUP BY nombre_evento
)

SELECT
    nombre_evento,
    usuarios_unicos,
    ROUND(
        100.0 * usuarios_unicos /
        LAG(usuarios_unicos) OVER (
            ORDER BY CASE nombre_evento
                WHEN 'first_visit' THEN 1
                WHEN 'add_to_cart' THEN 2
                WHEN 'select_item' THEN 3
                WHEN 'begin_checkout' THEN 4
                WHEN 'add_payment_info' THEN 5
                WHEN 'purchase' THEN 6
            END
        ),
        2
    ) AS tasa_conversion
FROM funnel
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'add_to_cart' THEN 2
    WHEN 'select_item' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,tasa_conversion
0,first_visit,7796,NaN
1,add_to_cart,7634,97.92
2,select_item,7582,99.32
3,begin_checkout,7208,95.07
4,add_payment_info,6250,86.71
5,purchase,6240,99.84


**🎯 Principal hallazgo**

El mayor punto de abandono está entre:

`begin_checkout → add_payment_info`

- Usuarios que llegan a checkout: 7,208
- Usuarios que agregan información de pago: 6,250
- Usuarios perdidos: 958
- Tasa de conversión: 86.71%
- Drop-off: 13.29%


Es claramente el mayor drop-off del funnel.


Esto apunta a que **el proceso de pago merece una investigación específica**: fricción en el formulario, problemas con métodos de pago, costos inesperados, errores técnicos, falta de confianza, etc. No podemos afirmar cuál es la causa solamente con estos datos.


**🏁 Conversión final**

También podemos calcular:
$ 6240 / 7796 ×100=80.04$%

Por lo tanto, la conversión final del funnel es aproximadamente 80.04% bajo esta metodología de usuarios únicos por evento.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [212]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [213]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [214]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),

actividad_semanal AS (
    SELECT
        id_usuario,
        MAX(CASE
            WHEN dias_despues_registro = 7 AND activo = 1 THEN 1
            ELSE 0
        END) AS retenido_w1,
        MAX(CASE
            WHEN dias_despues_registro = 14 AND activo = 1 THEN 1
            ELSE 0
        END) AS retenido_w2,
        MAX(CASE
            WHEN dias_despues_registro = 21 AND activo = 1 THEN 1
            ELSE 0
        END) AS retenido_w3
    FROM user_activity
    GROUP BY id_usuario
)

SELECT
    uc.cohorte,
    COUNT(DISTINCT uc.id_usuario) AS usuarios_iniciales,
    SUM(COALESCE(a.retenido_w1, 0)) AS retenido_w1,
    SUM(COALESCE(a.retenido_w2, 0)) AS retenido_w2,
    SUM(COALESCE(a.retenido_w3, 0)) AS retenido_w3,
    ROUND(
        100.0 * SUM(COALESCE(a.retenido_w1, 0))
        / COUNT(DISTINCT uc.id_usuario),
        2
    ) AS semana_1,
    ROUND(
        100.0 * SUM(COALESCE(a.retenido_w2, 0))
        / COUNT(DISTINCT uc.id_usuario),
        2
    ) AS semana_2,
    ROUND(
        100.0 * SUM(COALESCE(a.retenido_w3, 0))
        / COUNT(DISTINCT uc.id_usuario),
        2
    ) AS semana_3
FROM usuarios_cohorte uc
LEFT JOIN actividad_semanal a
    ON uc.id_usuario = a.id_usuario
GROUP BY uc.cohorte
ORDER BY uc.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---


## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  


In [215]:
# Cargar y explorar el dataset del experimento A/B
import pandas as pd

url_experiment = 'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'

experiment = pd.read_csv(url_experiment)

experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [216]:
# Revisar estructura, tipos de datos y valores faltantes
print(experiment.info())
print('\nValores faltantes:')
print(experiment.isna().sum())

print('\nValores únicos en convirtio:')
print(experiment['convirtio'].value_counts())

print('\nDistribución por variante:')
print(experiment['variante'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB
None

Valores faltantes:
id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64

Valores únicos en convirtio:
0    8401
1    1599
Name: convirtio, dtype: int64

Distribución por variante:
tratamiento    5035
control        4965
Name: variante, dtype: int64


In [217]:
# Calcular la tasa de conversión por variante
conversion_por_variante = experiment.groupby('variante')['convirtio'].agg(
    usuarios='count',
    conversiones='sum',
    tasa_conversion='mean'
)

conversion_por_variante['tasa_conversion'] = (
    conversion_por_variante['tasa_conversion'] * 100
).round(2)

conversion_por_variante

,usuarios,conversiones,tasa_conversion
variante,,,
control,4965,779,15.69
tratamiento,5035,820,16.29


---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión es igual entre la variante control y la variante tratamiento.
   - **H₁ (Hipótesis alternativa):** La tasa de conversión es diferente entre la variante control y la variante tratamiento.
   
**Test estadístico:** Prueba Z para comparación de dos proporciones.  
**Nivel de significancia alpha:** α = 0.05



In [218]:
# Test Z para comparar las tasas de conversión entre control y tratamiento

from statsmodels.stats.proportion import proportions_ztest

conversiones = [779, 820]
usuarios = [4965, 5035]

z_stat, p_value = proportions_ztest(conversiones, usuarios)

print(f"Estadístico Z: {z_stat:.4f}")
print(f"Valor p: {p_value:.6f}")

Estadístico Z: -0.8133
Valor p: 0.416059


**Recomendación para el jefe**

**No hay evidencia estadísticamente significativa de que la nueva UI del checkout mejore la tasa de conversión**. Aunque el tratamiento presenta una conversión ligeramente mayor (16.29% vs. 15.69%), la diferencia no es significativa (p = 0.4161). Por lo tanto, con este experimento no se recomienda implementar el cambio basándose únicamente en la mejora observada.

**Conclusión**: el experimento no demostró impacto estadísticamente significativo sobre la conversión.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# link de power bi o tableau
# link de one drive / google drive:
# https://drive.google.com/drive/folders/1cnwyKmQZ0Oh090rSyPb_xUK19J3TLOwM?usp=sharing